In [1]:
import os
from dotenv import load_dotenv

# .env 파일에서 환경 변수를 로드합니다.
load_dotenv()

# 환경 변수에서 API 키를 가져옵니다.
pinecone_api_key = os.getenv("PINECONE_API_KEY")

if pinecone_api_key:
    print("✅ Pinecone API 키가 성공적으로 로드되었습니다.")
    # print(f"API Key (first 5 chars): {pinecone_api_key[:5]}...") # 확인용 출력 (전체 키 출력 금지)
else:
    print("❌ Pinecone API 키를 찾을 수 없습니다. .env 파일을 확인해주세요.")

print("🎉 0단계 환경 설정이 완료되었습니다! 이제 데이터 준비를 시작할 수 있습니다.")

✅ Pinecone API 키가 성공적으로 로드되었습니다.
🎉 0단계 환경 설정이 완료되었습니다! 이제 데이터 준비를 시작할 수 있습니다.


In [2]:
import pandas as pd
from datasets import load_dataset
from bs4 import BeautifulSoup
from tqdm import tqdm
import json
import os

print("✅ 라이브러리 임포트 완료")

c:\Users\qwer8\anaconda3\envs\rag_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 라이브러리 임포트 완료


In [3]:
dataset = load_dataset("LGCNS/KorQuAD_2.0")
train = dataset["train"]
validation = dataset["validation"]

In [4]:
# 빠른 테스트를 위해 데이터셋의 일부만 사용합니다.
subset_size = 1000
train_subset = train.select(range(subset_size))

print(f"📄 데이터셋 로드 완료. 총 {len(train_subset)}개의 문서를 사용합니다.")

# 데이터 구조 확인 (첫 번째 데이터 예시)
# context 필드가 HTML 태그 없이 깨끗한 텍스트인 것을 확인합니다.
print("\n데이터 예시:")
print(train_subset[0])

📄 데이터셋 로드 완료. 총 1000개의 문서를 사용합니다.

데이터 예시:
{'title': '밀양_송전탑_사건', 'url': 'https://ko.wikipedia.org/wiki/밀양_송전탑_사건', 'context': '<!DOCTYPE html>\n<html>\n<head>\n<meta>\n<title>밀양 송전탑 사건 - 위키백과, 우리 모두의 백과사전</title>\n\n\n<link>\n\n<meta>\n<link>\n<meta>\n<meta>\n<meta>\n<meta>\n<link>\n<link>\n<link>\n<link>\n<link>\n<link>\n<link>\n<link>\n<link>\n<link>\n<link>\n\n</head>\n<body>\n<div></div>\n<div></div>\n<div>\n<a></a>\n<div></div>\n<div>\n</div>\n<h1>밀양 송전탑 사건</h1>\n<div>\n<div>위키백과, 우리 모두의 백과사전.</div>\n<div></div>\n<div></div>\n<a>둘러보기로 가기</a>\n<a>검색하러 가기</a>\n<div><div><table><tbody><tr><th colspan="2">밀양 송전탑 시위</th></tr><tr><th>날짜</th>\n<td>\n2008년 7월 ~ 현재</td></tr><tr><th>지역</th>\n<td>\n<span><a>대한민국</a> <a>경상남도</a> <a>밀양시</a></span></td></tr><tr><th>원인</th>\n<td>\n고압 송전선 설치 위치 문제</td></tr><tr><th>목적</th>\n<td>\n고압 송전선 설치 취소</td></tr><tr><th>종류</th>\n<td>\n<a>항의</a>, <a>시위</a></td></tr><tr><th>상태</th>\n<td>\n현재 시위 진행중</td></tr><tr><th colspan="2">시위 당사자</th></tr><tr><td colspan=

In [5]:
from bs4 import BeautifulSoup

corpus_chunks = []
print("\n⏳ 데이터 청킹 및 메타데이터 추가를 시작합니다 (HTML 파싱 포함)...")

# train_subset 변수를 사용합니다.
for doc_id, data in tqdm(enumerate(train_subset), total=len(train_subset)):
    context_html = data['context']
    if not isinstance(context_html, str) or not context_html:
        continue

    # 1. BeautifulSoup을 사용하여 HTML에서 순수 텍스트만 추출
    soup = BeautifulSoup(context_html, 'html.parser')
    pure_text = soup.get_text()
    
    # 2. 추출된 텍스트를 문단 기준으로 분할
    chunks = pure_text.strip().split("\n\n")
    
    # 3. 각 청크에 대한 메타데이터 생성
    for i, chunk_text in enumerate(chunks):
        cleaned_chunk = chunk_text.strip()
        if len(cleaned_chunk) < 20:
            continue
            
        chunk_info = {
            'chunk_id': f"doc_{doc_id}-chunk_{i}",
            'text': cleaned_chunk,
            'doc_id': doc_id,
            'title': data['title'],
            'source': f"LGCNS/KorQuAD_2.0 train-idx-{doc_id}",
            'url': data.get('url', 'N/A'),
            'section': "N/A",
            'revision_date': "2020-09-02",
            'language': "ko"
        }
        corpus_chunks.append(chunk_info)

print(f"\n✅ 청킹 완료! 총 {len(corpus_chunks)}개의 청크가 생성되었습니다.")

if corpus_chunks:
    print("\n[생성된 청크 데이터 예시 (HTML 제거됨)]")
    print(json.dumps(corpus_chunks[0], indent=2, ensure_ascii=False))


⏳ 데이터 청킹 및 메타데이터 추가를 시작합니다 (HTML 파싱 포함)...


100%|██████████| 1000/1000 [01:20<00:00, 12.47it/s]


✅ 청킹 완료! 총 52272개의 청크가 생성되었습니다.

[생성된 청크 데이터 예시 (HTML 제거됨)]
{
  "chunk_id": "doc_0-chunk_0",
  "text": "밀양 송전탑 사건 - 위키백과, 우리 모두의 백과사전",
  "doc_id": 0,
  "title": "밀양_송전탑_사건",
  "source": "LGCNS/KorQuAD_2.0 train-idx-0",
  "url": "https://ko.wikipedia.org/wiki/밀양_송전탑_사건",
  "section": "N/A",
  "revision_date": "2020-09-02",
  "language": "ko"
}


In [6]:
import pandas as pd
import os

# 데이터를 저장할 폴더가 없다면 생성합니다.
# 프로젝트 루트에서 실행하는 것을 기준으로 'data/' 경로를 사용합니다.
output_dir = 'data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 리스트를 Pandas DataFrame으로 변환
df_chunks = pd.DataFrame(corpus_chunks)

# Parquet 파일로 저장
output_path = os.path.join(output_dir, 'corpus_chunks.parquet')
df_chunks.to_parquet(output_path, index=False)

print(f"💾 데이터가 성공적으로 '{output_path}'에 저장되었습니다.")

# 저장된 데이터 확인
print("\n[저장된 DataFrame 정보]")
df_chunks.info()
print("\n[저장된 DataFrame 일부]")
print(df_chunks.head())

💾 데이터가 성공적으로 'data\corpus_chunks.parquet'에 저장되었습니다.

[저장된 DataFrame 정보]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52272 entries, 0 to 52271
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   chunk_id       52272 non-null  object
 1   text           52272 non-null  object
 2   doc_id         52272 non-null  int64 
 3   title          52272 non-null  object
 4   source         52272 non-null  object
 5   url            52272 non-null  object
 6   section        52272 non-null  object
 7   revision_date  52272 non-null  object
 8   language       52272 non-null  object
dtypes: int64(1), object(8)
memory usage: 3.6+ MB

[저장된 DataFrame 일부]
         chunk_id                                               text  doc_id  \
0   doc_0-chunk_0                      밀양 송전탑 사건 - 위키백과, 우리 모두의 백과사전       0   
1  doc_0-chunk_16                     둘러보기로 가기\n검색하러 가기\n밀양 송전탑 시위날짜       0   
2  doc_0-chunk_24                   밀